1️⃣ Idea para el esquema JSON objetivo

In [5]:
listado_repuestos_UN_vehículo ={
  "vehiculo": {
    "marca_vehiculo": "Toyota",
    "modelo": "Etios",
    "anio": 2013
  },
  "items": [
    {
      "descripcion_usuario": "pastillas de freno delanteras",
      "cantidad": 1
    },
    {
      "descripcion_usuario": "bomba de freno",
      "cantidad": 1
    }
  ],
  "notas": "algún comentario del usuario si hace falta"
}

2️⃣ clases con Pydantic para formatear el json

In [1]:
from pydantic import BaseModel, Field
from typing import List, Optional


class VehiculoPedido(BaseModel):
    # CAMPOS OBLIGATORIOS
    marca_vehiculo: str = Field(
        ...,
        description="Marca del vehículo (por ejemplo: Toyota, Renault, etc.)"
    )
    modelo: str = Field(
        ...,
        description="Modelo del vehículo (por ejemplo: Etios, Clio, etc.)"
    )

    # Campo opcional pero obligatorio si aparece (puede ser rango)
    anio: Optional[str] = Field(
        None,
        description="Año o rango de años del modelo (por ejemplo '2016' o '2016-2020')."
    )


class ItemPedido(BaseModel):
    descripcion_usuario: str = Field(
        ...,
        description="Descripción del repuesto que permita buscarlo en el catálogo"
    )
    cantidad: int = Field(
        1,
        description="Cantidad pedida (por defecto es 1)"
    )


class PedidoPorVehiculo(BaseModel):
    """
    Pedido asociado a UN vehículo concreto.
    """
    vehiculo: VehiculoPedido
    items: List[ItemPedido]


class PedidoRepuestos(BaseModel):
    """
    Pedido general, que puede incluir varios vehículos.
    """
    pedidos: List[PedidoPorVehiculo]
    notas: Optional[str] = Field(
        None,
        description="Comentarios generales (plazos, entrega, notas del cliente, etc.)"
    )


3️⃣ Prompt para que el LLM haga SOLO extracción y devuelva JSON

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt_pedido = ChatPromptTemplate.from_messages([
    (
        "system",
        "Sos un asistente especializado en interpretar pedidos de repuestos escritos en lenguaje natural. "
        "El usuario puede pedir repuestos para uno o varios vehículos en el mismo texto. "
        "Tu tarea es identificar cada vehículo mencionado (marca, modelo y, si aparece, año) y asociar "
        "los repuestos correspondientes a ese vehículo.\n\n"

        "La salida debe respetar EXCLUSIVAMENTE el modelo Pydantic 'PedidoRepuestos', cuyo esquema es:\n"
        "- pedidos: lista de objetos, cada uno con:\n"
        "    - vehiculo: con campos marca_vehiculo (str), modelo (str), anio (str opcional)\n"
        "    - items: lista de repuestos del vehículo. Cada item tiene:\n"
        "         - descripcion_usuario (str, obligatorio)\n"
        "         - cantidad (int, por defecto 1)\n"
        "- notas: texto opcional con comentarios generales.\n\n"

        "REGLAS IMPORTANTES:\n"
        "1. No inventes marca ni modelo. Si no pueden determinarse por el texto, devolvé un error indicando qué falta.\n"
        "2. Si hay varios vehículos, generá una entrada independiente por cada uno dentro de la lista 'pedidos'.\n"
        "3. Extraé cantidades cuando sea posible. Si no están explícitas, usar 1.\n"
        "4. No agregues campos que no existan en el modelo.\n"
        "5. Si hay texto adicional (plazos, envío, comentarios), colocarlo en 'notas'.\n"
        "6. Agrupá repuestos por vehículo correctamente.\n"
        "7. Devolvé SOLO un objeto compatible con PedidoRepuestos."
    ),
    (
        "human",
        "Texto del pedido:\n\n{texto_pedido}"
    )
])

In [7]:
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableSequence

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# Le decimos al LLM que queremos la salida como PedidoRepuestos
llm_struct = llm.with_structured_output(PedidoRepuestos)


chain = prompt_pedido | llm_struct

def parsear_pedido(texto_pedido: str) -> PedidoRepuestos:
    try:
        return chain.invoke({"texto_pedido": texto_pedido})
    except Exception as e:
        print("⚠️ Error al interpretar el pedido:", e)
        raise


4️⃣ Uso con with_structured_output (LangChain + Groq)

In [10]:
texto = """
Necesito para un Toyota Etios 2016: pastillas de freno delanteras y una bomba de freno.
También estoy necesitando un juego de amortiguadores traseros para un Renault Clío 2018.
Por favor, entregarlo en el taller antes del viernes.
"""

pedido = parsear_pedido(texto)

print(pedido.model_dump_json(indent=2))

{
  "pedidos": [
    {
      "vehiculo": {
        "marca_vehiculo": "Toyota",
        "modelo": "Etios",
        "anio": "2016"
      },
      "items": [
        {
          "descripcion_usuario": "pastillas de freno delanteras",
          "cantidad": 1
        },
        {
          "descripcion_usuario": "bomba de freno",
          "cantidad": 1
        }
      ]
    },
    {
      "vehiculo": {
        "marca_vehiculo": "Renault",
        "modelo": "Clio",
        "anio": "2018"
      },
      "items": [
        {
          "descripcion_usuario": "juego de amortiguadores traseros",
          "cantidad": 1
        }
      ]
    }
  ],
  "notas": "Entregar antes del viernes"
}
